# Chapter 18 - Entropy Features

## Preparation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

import random

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
from scipy.stats import norm, moment
import statsmodels.api as sm

from utils.sampling_bars import dollar_bar
from utils.entropy import binary_encoding, quantile_encoding, sigma_encoding, plug_in_entropy_estimation, kontoyiannis_entropy_estimation

%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 16,6

data_path = '../data/processed/clean_IVE_tickbidask.parq'
df = pd.read_parquet(data_path)

dollar_bars = dollar_bar(df)

## 1. Form dollar bars on E-mini S&P 500 futures:

In [ ]:
raw_returns = dollar_bars["price"].diff().dropna()

### a. Quantize the returns series using the binary method.

In [ ]:
binary_encoded_returns = binary_encoding(raw_returns)

# count the number of each distinct values
pd.Series(binary_encoded_returns).value_counts()

### b. Quantize the returns series using the quantile encoding, using 10 letters.

In [ ]:
quantile_encoded_returns = quantile_encoding(raw_returns, n_quantiles=10)

# count the number of each distinct values
pd.Series(quantile_encoded_returns).value_counts()

### c. Quantize the returns series using the sigma encoding, where $\sigma$ is the standard deviation of all bar returns.

In [ ]:
sigma_encoded_returns = sigma_encoding(raw_returns, raw_returns.std())

# count the number of each distinct values
pd.Series(sigma_encoded_returns).value_counts()

### d. Compute the entropy of the three encoded series, using the plug-in method.

In [ ]:
binary_plugin_entropy = plug_in_entropy_estimation(binary_encoded_returns, 10)
quantile_plugin_entropy = plug_in_entropy_estimation(quantile_encoded_returns, 10)
sigma_plugin_entropy = plug_in_entropy_estimation(sigma_encoded_returns, 10)

print(f"Binary plugin entropy: {binary_plugin_entropy}")
print(f"Quantile plugin entropy: {quantile_plugin_entropy}")
print(f"Sigma plugin entropy: {sigma_plugin_entropy}")

It is normal that the quantile encoding obtains the highest entropy, as the uniform or close to uniform distribution of codes tends to increase entropy readings on average.

### e. Compute the entropy of the three encoded series, using Kontoyiannis’ method, with a window size of 100.

In [ ]:
binary_konto_entropy = kontoyiannis_entropy_estimation(binary_encoded_returns, 100)
quantile_konto_entropy = kontoyiannis_entropy_estimation(quantile_encoded_returns, 100)
sigma_konto_entropy = kontoyiannis_entropy_estimation(sigma_encoded_returns, 100)

print(f"Binary konto entropy: {binary_konto_entropy['entropy']}")
print(f"Quantile konto entropy: {quantile_konto_entropy['entropy']}")
print(f"Sigma konto entropy: {sigma_konto_entropy['entropy']}")


## 2. Using the bars from exercise 1:

### a. Compute the returns series, $\{r_t\}$.

### b. Encode the series as follows: 0 if $r_tr_{t-1} < 0$, and 1 if $r_tr_{t-1} \geq 0$.

In [ ]:
lagged_returns = raw_returns.shift(1)
multiplied_returns = raw_returns * lagged_returns
multiplied_returns = multiplied_returns.dropna()

binary_encoded_multiplied_returns = np.where(multiplied_returns < 0, 0, 1)

# count the number of each distinct values
pd.Series(binary_encoded_multiplied_returns).value_counts()

### c. Partition the series into 1000 non-overlapping subsets of equal size (you  may have to drop some observations at the beginning).

In [ ]:
num_subsets = 1000
subset_size = len(binary_encoded_multiplied_returns) // num_subsets
num_to_drop = len(binary_encoded_multiplied_returns) - subset_size * num_subsets

# Drop observations from the beginning so the remaining sample splits exactly into 1000 blocks.
binary_encoded_multiplied_returns_trimmed = binary_encoded_multiplied_returns[num_to_drop:]
binary_encoded_multiplied_returns_subsets = binary_encoded_multiplied_returns_trimmed.reshape(
    num_subsets,
    subset_size,
)

print(f"Dropped observations: {num_to_drop}")
print(f"Subset array shape: {binary_encoded_multiplied_returns_subsets.shape}")


### d. Compute the entropy of each of the 1000 encoded subsets, using the plug-in method with word length 10.


In [ ]:
plugin_word_length_subsets = 10
plugin_entropy_by_subset = pd.Series(
    [
        plug_in_entropy_estimation(subset, plugin_word_length_subsets)
        for subset in binary_encoded_multiplied_returns_subsets
    ],
    name="plug_in_entropy",
)

plugin_entropy_by_subset.describe()


### e. Compute the entropy of each of the 1000 encoded subsets, using the  Kontoyiannis method, with a window size of 100.

In [ ]:
konto_entropy_by_subset = pd.Series(
    [kontoyiannis_entropy_estimation(subset, 100)["entropy"] for subset in binary_encoded_multiplied_returns_subsets],
    name="kontoyiannis_entropy",
)

konto_entropy_by_subset.describe()


### f. Compute the correlation between results 2.d and 2.e.


In [ ]:
subset_entropy_results = pd.concat(
    [plugin_entropy_by_subset, konto_entropy_by_subset],
    axis=1,
)
subset_entropy_correlation = subset_entropy_results.corr().loc[
    "plug_in_entropy",
    "kontoyiannis_entropy",
]

print(f"Correlation between plug-in and Kontoyiannis subset entropies: {subset_entropy_correlation}")
subset_entropy_results.describe()


## 3. Draw 1000 observations from a standard Normal distribution:

In [ ]:
rng = np.random.default_rng(42)
observations = rng.normal(0, 1, 1000)


### a. What is the true entropy of this process?

The differential entropy of an IID $N(0,1)$ process is
$$h(X)=\frac{1}{2}\log(2\pi e\sigma^2).$$
The base of the logarithm determines the units. The helper functions in `utils.entropy` use `log2`, so the comparable continuous-entropy value is reported in bits. After quantile encoding into 8 equally populated labels, the true marginal discrete entropy of the encoded IID process is $\log_2(8)=3$ bits per observation.


In [ ]:
true_entropy_normal_nats = 0.5 * np.log(2 * np.pi * np.e)
true_entropy_normal_bits = 0.5 * np.log2(2 * np.pi * np.e)
true_entropy_8_quantile_labels_bits = np.log2(8)

print(f"Differential entropy of N(0, 1): {true_entropy_normal_nats:.6f} nats")
print(f"Differential entropy of N(0, 1): {true_entropy_normal_bits:.6f} bits")
print(f"Entropy of the 8-quantile encoded IID process: {true_entropy_8_quantile_labels_bits:.6f} bits")


### b. Label the observations according to 8 quantiles.

In [ ]:
quantile_encoded_observations = quantile_encoding(observations, n_quantiles=8)

print(pd.Series(quantile_encoded_observations).value_counts())

### c. Estimate the entropy using the plug-in method with word length 2.


In [ ]:
plug_in_word_length_quantile = 2
quantile_plugin_entropy_normal = plug_in_entropy_estimation(
    quantile_encoded_observations,
    plug_in_word_length_quantile,
)
print(f"Entropy of the Normal labels estimated by plug-in method: {quantile_plugin_entropy_normal}")


### d. Estimate the entropy using the Kontoyiannis method:

#### i. using a window size of 10.

In [ ]:
quantile_konto_entropy_normal_10 = kontoyiannis_entropy_estimation(quantile_encoded_observations, 10)
print(
    "Entropy of the Normal labels estimated by Kontoyiannis method "
    f"with window size 10: {quantile_konto_entropy_normal_10['entropy']}"
)


#### ii. using a window size of 100.

In [ ]:
quantile_konto_entropy_normal_100 = kontoyiannis_entropy_estimation(quantile_encoded_observations, 100)
print(
    "Entropy of the Normal labels estimated by Kontoyiannis method "
    f"with window size 100: {quantile_konto_entropy_normal_100['entropy']}"
)


## 4. Using the draws from exercise 3, $\{x_t\}_{t = 1, \ldots 1000}$:

### a. Compute $y_t = \rho y_{t − 1} + x_t$, where $\rho = 0.5, y_0 = 0$.

In [ ]:
rho = 0.5
num_obs = 1000

x_t = observations
kernel = rho ** np.arange(num_obs)
y_t = np.convolve(x_t, kernel, mode="full")[:num_obs]

pd.Series(y_t, name="y_t").head()


### b. Label $\{y_t\}$ the observations according to 8 quantiles.

In [ ]:
quantile_encoded_y_t = quantile_encoding(y_t, n_quantiles=8)

print(pd.Series(quantile_encoded_y_t).value_counts())

### c. Estimate the entropy using the plug-in method with the same word length as exercise 3.


In [ ]:
quantile_plugin_entropy_y_t = plug_in_entropy_estimation(
    quantile_encoded_y_t,
    plug_in_word_length_quantile,
)
print(f"Entropy of $y_t$ estimated by plug-in method: {quantile_plugin_entropy_y_t}")


### d. Estimate the entropy using the Kontoyiannis method

#### i. using a window size of 10.

In [ ]:
quantile_konto_entropy_y_t_10 = kontoyiannis_entropy_estimation(quantile_encoded_y_t, 10)
print(
    "Entropy of $y_t$ estimated by Kontoyiannis method "
    f"with window size 10: {quantile_konto_entropy_y_t_10['entropy']}"
)


#### ii. using a window size of 100.

In [ ]:
quantile_konto_entropy_y_t_100 = kontoyiannis_entropy_estimation(quantile_encoded_y_t, 100)
print(
    "Entropy of $y_t$ estimated by Kontoyiannis method "
    f"with window size 100: {quantile_konto_entropy_y_t_100['entropy']}"
)


## 5. Suppose a portfolio of 10 holdings with equal dollar allocations.

### a. The portion of the total risk contributed by the ith principal component is $\frac{1}{10}, i = 1, \ldots, 10$. What is the portfolio's entropy?

In [ ]:
equal_risk_contrib = np.repeat(1 / 10, 10)
portfolio_entropy_5a = -np.sum(equal_risk_contrib * np.log(equal_risk_contrib))

print(f"Portfolio entropy: {portfolio_entropy_5a}")


### b. The portion of the total risk contributed by the ith principal component is $\frac{11-i}{55}, i = 1, \ldots, 10$. What is the portfolio's entropy?


In [ ]:
linear_risk_contrib = (11 - np.arange(1, 11)) / 55
portfolio_entropy_5b = -np.sum(linear_risk_contrib * np.log(linear_risk_contrib))

print(f"Risk contribution sum: {linear_risk_contrib.sum()}")
print(f"Portfolio entropy: {portfolio_entropy_5b}")


### c. The portion of the total risk contributed by the ith principal component is $\alpha \frac{1}{10} + (1 - \alpha)\frac{11-i}{55}, i = 1, \ldots, 10, \alpha \in [0,1]$. Plot the portfolio's entropy as a function of $\alpha$.


In [ ]:
alphas = np.linspace(0, 1, 100)
equal_risk_contrib = np.repeat(1 / 10, 10)
linear_risk_contrib = (11 - np.arange(1, 11)) / 55

risk_contrib_by_alpha = (
    alphas[:, None] * equal_risk_contrib
    + (1 - alphas[:, None]) * linear_risk_contrib
)
entropies = -np.sum(risk_contrib_by_alpha * np.log(risk_contrib_by_alpha), axis=1)

plt.plot(alphas, entropies)
plt.xlabel(r"$\alpha$")
plt.ylabel("Portfolio entropy")
plt.title(r"Portfolio entropy as a function of $\alpha$")
plt.show()
